# AoI-Optimal UAV Relay Scheduling — Separate Figures
Produces **6 individual files**: `fig_a.pdf/png` … `fig_e.pdf/png` in `./figures/`

In [1]:
#!/usr/bin/env python3
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator, AutoMinorLocator
from scipy.ndimage import uniform_filter1d
from scipy import stats
from multiprocessing import Pool, cpu_count
from dataclasses import dataclass, replace, field
import os, warnings, time, json
warnings.filterwarnings('ignore')

# ── Font sizes (standalone figures → bigger than combined) ──────────────
FS = dict(base=22, label=24, title=23, tick=21, legend=19, annot=20, note=17)

IEEE_RC = {
    'font.family': 'DejaVu Serif', 'font.size': FS['base'],
    'axes.labelsize': FS['label'], 'axes.titlesize': FS['title'],
    'axes.titleweight': 'bold',    'axes.labelweight': 'bold',
    'xtick.labelsize': FS['tick'], 'ytick.labelsize': FS['tick'],
    'legend.fontsize': FS['legend'], 'legend.framealpha': 0.93,
    'legend.edgecolor': '#AAAAAA',  'legend.handlelength': 2.8,
    'pdf.fonttype': 42, 'figure.dpi': 150,
}

def _sax(ax):
    ax.set_facecolor('white')
    ax.grid(True, 'major', color='#DDDDDD', lw=0.9, zorder=0)
    ax.grid(True, 'minor', color='#EEEEEE', lw=0.4, zorder=0)
    ax.set_axisbelow(True)
    for s in ['top','right']:   ax.spines[s].set_visible(False)
    for s in ['left','bottom']: ax.spines[s].set_linewidth(1.6); ax.spines[s].set_color('#333')
    ax.tick_params(which='both', direction='in', width=1.3, labelsize=FS['tick'])
    ax.xaxis.set_minor_locator(AutoMinorLocator(2))
    ax.yaxis.set_minor_locator(AutoMinorLocator(2))

def _save(fig, outdir, name):
    os.makedirs(outdir, exist_ok=True)
    for ext in ('.pdf', '.png'):
        fig.savefig(os.path.join(outdir, name+ext), dpi=200,
                    bbox_inches='tight', facecolor='white')
    plt.close(fig)
    print(f'  ✓ {name}.pdf / .png  →  {outdir}/')

print('Imports OK')

Imports OK


In [2]:
# ══════════════════════════════════════════════════════════════════
# CONFIG
# ══════════════════════════════════════════════════════════════════
@dataclass
class Cfg:
    seed:           int   = 42
    n_trials:       int   = 50
    n_veh:          int   = 8
    road_len:       float = 600.0
    t_sim:          float = 60.0
    dt:             float = 0.1
    bsm_rate:       float = 10.0
    direct_range:   float = 300.0
    k_relay:        int   = 6
    h_uav:          float = 60.0
    uav_x:          float = 300.0
    relay_lat:      float = 0.005
    pdr_min:        float = 0.10
    safety_thr:     float = 0.5
    f_c:            float = 5.9e9
    p_tx_dbm:       float = 23.0
    nf_db:          float = 9.0
    bw_mhz:         float = 10.0
    h_ant:          float = 1.5
    snr_mid:        float = 8.0
    snr_slope:      float = 3.5
    nak_v2v:        float = 1.5
    nak_u2v:        float = 3.0
    sigma_v:        float = 1.2
    v_rel:          float = 15.0
    t_react:        float = 0.1
    age_thr:        float = 0.30
    scale_nv:       tuple = (4, 6, 8, 10, 12, 14)
    scale_tr:       int   = 15
    scale_tsim:     float = 30.0
    dataset:        str   = 'dataset.txt'
    outdir:         str   = 'figures'
    n_workers:      int   = -1
    jitter_std:     float = 0.02  # MAC timing jitter std-dev (s); 0 = no jitter

    def __post_init__(self):
        if self.n_workers < 0:
            self.n_workers = min(8, cpu_count())

    @property
    def T(self):     return int(self.t_sim / self.dt)
    @property
    def bstep(self): return max(1, int(1.0 / (self.bsm_rate * self.dt)))
    @property
    def aoi_min(self): return 1.0 / self.bsm_rate + self.relay_lat
    @property
    def r2(self):    return (self.v_rel * self.t_react) ** 2
    @property
    def lam(self):   return 3e8 / self.f_c
    @property
    def ktb(self):   return -174.0 + 10*np.log10(self.bw_mhz*1e6) + self.nf_db
    @property
    def dbp(self):   return 4 * self.h_ant**2 / self.lam

def pair_idx(n):
    pi = [i for i in range(n) for j in range(n) if i != j]
    pj = [j for i in range(n) for j in range(n) if i != j]
    return np.array(pi), np.array(pj)

print('Config OK')

Config OK


In [3]:
# ══════════════════════════════════════════════════════════════════
# CHANNEL
# ══════════════════════════════════════════════════════════════════
def pl_v2v(d, c):
    d = np.maximum(d, 0.5)
    return np.where(d <= c.dbp,
        20*np.log10(4*np.pi*d/c.lam),
        40*np.log10(d) - 20*np.log10(c.h_ant**2))

def pl_u2v(dh, c):
    d3 = np.sqrt(np.maximum(dh, 0.5)**2 + c.h_uav**2)
    return 20*np.log10(4*np.pi*d3/c.lam)

def snr_v2v(d, c): return c.p_tx_dbm - pl_v2v(d, c) - c.ktb
def snr_u2v(d, c): return c.p_tx_dbm - pl_u2v(d, c) - c.ktb

def pdr_fn(snr, c):
    return np.clip(0.5*(1+np.tanh((snr-c.snr_mid)/c.snr_slope)), 0, 1)

def nak_db(m, sz, rng):
    return 10*np.log10(np.maximum(rng.gamma(m, 1/m, size=sz), 1e-15))

def epoch_ch(pos, duav, PI, PJ, c, rng):
    dp   = np.linalg.norm(pos[PI]-pos[PJ], axis=1)
    sv   = snr_v2v(dp, c)  + nak_db(c.nak_v2v, len(PI), rng)
    fu   = nak_db(c.nak_u2v, c.n_veh, rng)
    sui  = snr_u2v(duav[PI], c) + fu[PI]
    suj  = snr_u2v(duav[PJ], c) + fu[PJ]
    pd   = pdr_fn(sv,  c)
    pr   = pdr_fn(sui, c) * pdr_fn(suj, c)
    direct = (dp <= c.direct_range) & (pd >= c.pdr_min)
    return pd, pr, direct

print('Channel OK')

Channel OK


In [4]:
# ══════════════════════════════════════════════════════════════════
# MOBILITY  (NGSIM if available, else synthetic IDM-like)
# ══════════════════════════════════════════════════════════════════
_NGSIM_DF = None
_HAS_REAL = False

def load_ngsim(path):
    cols = ['Dataset_ID','Frame_ID','Vehicle_ID','Global_Time',
            'Local_X','Local_Y','Global_X','Global_Y',
            'v_Length','v_Width','v_Class','v_Vel','v_Acc',
            'Lane_ID','Preceding','Following','Space_Hdwy']
    df = pd.read_csv(path, sep=r'\s+', header=None, names=cols, usecols=range(17))
    df['Local_X'] *= 0.3048
    df['Local_Y'] *= 0.3048
    return df

def ngsim_pos(df, trial_idx, c, rng):
    wf         = max(c.T // 2, 50)
    all_frames = sorted(df['Frame_ID'].unique())
    nf         = len(all_frames)
    start      = (trial_idx * wf) % max(1, nf - wf)
    win        = df[df['Frame_ID'].isin(all_frames[start:start+wf])]
    vc         = win['Vehicle_ID'].value_counts()
    good       = vc[vc >= int(0.8*wf)].index.tolist()
    pool       = good if len(good) >= c.n_veh else vc.index.tolist()
    chosen     = rng.choice(pool, size=c.n_veh, replace=False)
    pos        = np.zeros((c.T, c.n_veh, 2))
    for k, vid in enumerate(chosen):
        row  = win[win['Vehicle_ID']==vid].sort_values('Frame_ID')
        xc   = row['Local_Y'].values; yc = row['Local_X'].values
        L    = min(len(xc), wf)
        rep  = int(np.ceil(c.T / L))
        pos[:, k, 0] = np.tile(xc[:L], rep)[:c.T]
        pos[:, k, 1] = np.tile(yc[:L], rep)[:c.T]
    return pos

def synth_pos(seed, c):
    rng  = np.random.default_rng(seed)
    spd  = rng.normal(25.0, 5.0, c.n_veh).clip(10.0, 40.0)
    x0   = rng.uniform(0, c.road_len, c.n_veh)
    lane = np.where(np.arange(c.n_veh) < c.n_veh//2, -1.875, 1.875)
    acc  = rng.normal(0, 0.3, (c.T, c.n_veh)) * c.dt
    spd_t= np.clip(spd[None,:] + np.cumsum(acc, axis=0), 5, 45)
    xs   = (x0 + np.cumsum(spd_t * c.dt, axis=0)) % c.road_len
    ys   = np.broadcast_to(lane, (c.T, c.n_veh)).copy()
    pos  = np.empty((c.T, c.n_veh, 2))
    pos[:,:,0] = xs; pos[:,:,1] = ys
    return pos

print('Mobility OK')

Mobility OK


In [5]:
# ══════════════════════════════════════════════════════════════════
# POLICIES & LABELS
# ══════════════════════════════════════════════════════════════════
MODES  = ['no_relay','random','round_robin','tput_opt','age_threshold','aoi_opt']
LABELS = {'no_relay':'No Relay','random':'Random','round_robin':'Round-Robin',
          'tput_opt':'Tput-Opt','age_threshold':'Age-Threshold','aoi_opt':'AoI-Opt (Ours)'}
COLORS = {'no_relay':'#C62828','random':'#E65100','round_robin':'#6A1B9A',
          'tput_opt':'#F57F17','age_threshold':'#00838F','aoi_opt':'#0D47A1'}
STYLES = {'no_relay':(2.5,'-',0.80),'random':(2.5,'-.',0.85),
          'round_robin':(2.5,':',0.85),'tput_opt':(3.2,'--',1.0),
          'age_threshold':(2.8,'-.',0.90),'aoi_opt':(4.0,'-',1.0)}

def select_relay(mode, elig, aoi, pdr_r, c, rpick, rr):
    ne = int(elig.sum())
    if ne == 0: return np.empty(0,int), rr
    ei = np.where(elig)[0]; ks = min(c.k_relay, ne)
    if mode == 'random':
        return ei[np.argsort(rpick[ei])[:ks]], rr
    if mode == 'round_robin':
        rr %= ne
        return ei[[(rr+i)%ne for i in range(ks)]], (rr+ks)%ne
    if mode == 'tput_opt':
        sc = np.where(elig, pdr_r, -1.)
        tk = np.argpartition(sc,-ks)[-ks:]
        return tk[sc[tk]>0], rr
    if mode == 'age_threshold':
        ab = elig & (aoi > c.age_thr)
        pool = ab if ab.any() else elig
        sc = np.where(pool, pdr_r, -1.)
        kp = min(c.k_relay, int(pool.sum()))
        tk = np.argpartition(sc,-kp)[-kp:]
        return tk[sc[tk]>0], rr
    if mode == 'aoi_opt':
        red = np.maximum(0., aoi - c.aoi_min)
        sc  = np.where(elig, pdr_r*red, -1.)
        tk  = np.argpartition(sc,-ks)[-ks:]
        tk  = tk[sc[tk]>0]
        return tk[np.argsort(-pdr_r[tk], kind='stable')], rr
    return np.empty(0,int), rr

print('Policies OK')

Policies OK


In [6]:
# ══════════════════════════════════════════════════════════════════
# TRIAL RUNNER
# ══════════════════════════════════════════════════════════════════
def run_trial(args):
    seed, mode, tidx, store, c = args
    rng = np.random.default_rng(seed)
    PI, PJ = pair_idx(c.n_veh); P = len(PI)

    if _HAS_REAL and _NGSIM_DF is not None:
        pos = ngsim_pos(_NGSIM_DF, tidx, c, rng)
    else:
        pos = synth_pos(seed, c)

    uav  = np.array([c.uav_x, 0.])
    duav = np.linalg.norm(pos - uav[None,None,:], axis=2)
    epochs = np.arange(0, c.T, c.bstep); ne = len(epochs)
    rd = rng.random((ne, P)); rr_ = rng.random((ne, c.k_relay)); rp = rng.random((ne, P))

    aoi   = np.full(P, c.aoi_min)
    aoi_h = np.empty((c.T, P), np.float32)
    deliv = np.zeros(P, np.int32); rrp = 0

    for t in range(c.T):
        aoi += c.dt; aoi_h[t] = aoi
        if t % c.bstep != 0: continue
        ep = t // c.bstep
        if ep >= ne: continue
        pd, pr, direct = epoch_ch(pos[t], duav[t], PI, PJ, c, rng)
        hits = direct & (rd[ep] < pd)
        if hits.any():
            rng_jit = rng.normal(0, c.jitter_std, np.sum(hits))
            aoi[hits] = np.clip(c.aoi_min - c.relay_lat + rng_jit, 0, np.inf)
        deliv[hits] += 1
        if mode != 'no_relay':
            elig = (~direct) & (pr >= c.pdr_min)
            tk, rrp = select_relay(mode, elig, aoi, pr, c, rp[ep], rrp)
            if len(tk):
                tk = tk[:c.k_relay]; nr = len(tk)
                hit = rr_[ep,:nr] < pr[tk]
                if hit.any():
                    rng_jit = rng.normal(0, c.jitter_std, np.sum(hit))
                    aoi[tk[hit]] = np.clip(c.aoi_min + rng_jit, 0, np.inf)
                    deliv[tk[hit]] += 1

    risk_h = 1 - np.exp(-(c.sigma_v**2 * aoi_h.astype(float)**2)/(2*c.r2))
    dr     = deliv.sum()/(P*ne) if ne>0 else 0.
    mp     = aoi_h.mean(0)
    jain   = mp.sum()**2 / (len(mp)*(mp**2).sum()+1e-15)
    out = dict(mean_aoi=float(aoi_h.mean()), peak_p95=float(np.percentile(aoi_h,95)),
               frac_over=float((aoi_h>c.safety_thr).mean()), mean_risk=float(risk_h.mean()),
               bsm_dr=float(dr), jain=float(jain),
               aoi_tr=aoi_h.mean(1), risk_tr=risk_h.mean(1).astype(np.float32))
    if store:
        out['peak_aoi'] = np.percentile(aoi_h, 95, axis=0).astype(np.float32)
    return out

print('Trial runner OK')

Trial runner OK


In [7]:
# ══════════════════════════════════════════════════════════════════
# SIMULATION RUNNERS
# ══════════════════════════════════════════════════════════════════
def run_all(c):
    rng   = np.random.default_rng(c.seed)
    seeds = rng.integers(0, 100_000, c.n_trials).tolist()
    res   = {}
    for mode in MODES:
        args = [(int(s), mode, i, False, c) for i,s in enumerate(seeds)]
        with Pool(c.n_workers) as pool: results = pool.map(run_trial, args)
        vals = np.array([r['mean_aoi'] for r in results])
        mid  = int(np.argmin(np.abs(vals - np.median(vals))))
        results[mid] = run_trial((int(seeds[mid]), mode, mid, True, c))
        res[mode] = results
        print(f"  {LABELS[mode]:22s}  AoI={vals.mean():.4f}±{vals.std(ddof=1):.4f}  "
              f"Risk={np.mean([r['mean_risk'] for r in results]):.4f}  "
              f"DR={np.mean([r['bsm_dr'] for r in results])*100:.1f}%")
    return res

def run_scale(c):
    print('\nScalability sweep...')
    t95 = stats.t.ppf(0.975, c.scale_tr-1)
    sc  = {}
    for nv in c.scale_nv:
        cs = replace(c, n_veh=nv, n_trials=c.scale_tr, t_sim=c.scale_tsim)
        rng_s   = np.random.default_rng(c.seed + nv*7919)
        seeds_s = rng_s.integers(0, 100_000, c.scale_tr).tolist()
        sc[nv]  = {}
        for mode in ['tput_opt','age_threshold','aoi_opt']:
            args = [(int(s), mode, i, False, cs) for i,s in enumerate(seeds_s)]
            with Pool(c.n_workers) as pool: tr = pool.map(run_trial, args)
            v = np.array([r['mean_aoi'] for r in tr])
            sc[nv][mode] = (float(v.mean()), float(t95*v.std(ddof=1)/np.sqrt(len(v))))
        ao = sc[nv]['aoi_opt'][0]; tp = sc[nv]['tput_opt'][0]
        print(f"  N={nv:2d}: AoI-Opt={ao:.4f}s  Tput-Opt={tp:.4f}s  gain={100*(tp-ao)/tp:.1f}%")
    return sc

def summarise(res, c):
    S = {}; t95 = stats.t.ppf(0.975, c.n_trials-1)
    for mode in MODES:
        S[mode] = {}
        for k in ['mean_aoi','peak_p95','frac_over','mean_risk','bsm_dr','jain']:
            v = np.array([r[k] for r in res[mode]])
            S[mode][k] = (float(v.mean()), float(t95*v.std(ddof=1)/np.sqrt(c.n_trials)), v)
    for cmp in ['tput_opt','age_threshold']:
        for k in ['mean_aoi','mean_risk']:
            a = np.array([r[k] for r in res['aoi_opt']])
            b = np.array([r[k] for r in res[cmp]])
            t, p = stats.ttest_rel(a, b)
            S[f't_{k}_vs_{cmp}'] = (float(t), float(p))
    for m in ['mean_aoi','mean_risk']:
        ref = S['tput_opt'][m][0]; our = S['aoi_opt'][m][0]
        S[f'red_{m}'] = 100*(ref-our)/ref if ref>0 else 0.
    return S

print('Runners OK')

def run_sensitivity(c):
    """Sweep UAV altitude × K_relay → mean AoI for aoi_opt/tput_opt/age_threshold."""
    print('\nSensitivity sweep (alt × K)...')
    alts = [30, 60, 120]
    ks   = [4, 6, 8]
    sens = {}
    for alt in alts:
        for k in ks:
            key = f'h{int(alt)}_K{k}'
            cs  = replace(c, h_uav=alt, k_relay=k, n_trials=20, t_sim=30.0)
            rng_s = np.random.default_rng(c.seed + int(alt)*31 + k*97)
            seeds_s = rng_s.integers(0, 100_000, cs.n_trials).tolist()
            entry = {}
            for mode in ['tput_opt', 'age_threshold', 'aoi_opt']:
                args = [(int(s), mode, i, False, cs) for i, s in enumerate(seeds_s)]
                with Pool(c.n_workers) as pool:
                    tr = pool.map(run_trial, args)
                v = np.array([r['mean_aoi'] for r in tr])
                entry[mode] = (float(v.mean()), float(v.std(ddof=1)))
            sens[key] = entry
            print(f'  {key}: AoI-Opt={entry["aoi_opt"][0]:.4f}  Tput={entry["tput_opt"][0]:.4f}')
    return sens


def run_ngsim_robust(c):
    """Sweep NGSIM smoothing window → AoI robustness check."""
    global _NGSIM_DF, _HAS_REAL
    if not _HAS_REAL or _NGSIM_DF is None:
        print('  (no NGSIM data — skipping robustness sweep)')
        return {}
    print('\nNGSIM robustness sweep (smoothing windows)...')
    original_df = _NGSIM_DF.copy()
    robust = {}
    for sw in [5, 10, 20]:
        df_s = original_df.copy()
        df_s['Local_X'] = uniform_filter1d(df_s['Local_X'].values, size=sw)
        df_s['Local_Y'] = uniform_filter1d(df_s['Local_Y'].values, size=sw)
        _NGSIM_DF = df_s
        rng_r = np.random.default_rng(c.seed + sw * 113)
        seeds_r = rng_r.integers(0, 100_000, 20).tolist()
        cs = replace(c, n_trials=20, t_sim=30.0)
        entry = {}
        for mode in ['tput_opt', 'aoi_opt']:
            args = [(int(s), mode, i, False, cs) for i, s in enumerate(seeds_r)]
            with Pool(c.n_workers) as pool:
                tr = pool.map(run_trial, args)
            v = np.array([r['mean_aoi'] for r in tr])
            entry[mode] = (float(v.mean()), float(v.std(ddof=1)))
        robust[f'sw{sw}'] = entry
        print(f'  sw={sw}: AoI-Opt={entry["aoi_opt"][0]:.4f}  Tput={entry["tput_opt"][0]:.4f}')
    _NGSIM_DF = original_df  # restore
    return robust


def run_jitter_sweep(c):
    """Sweep jitter_std → mean AoI to bound optimality gap."""
    print('\nJitter sweep...')
    jits = [0.0, 0.01, 0.02, 0.05]
    jit_res = {}
    for jv in jits:
        cs = replace(c, jitter_std=jv, n_trials=20, t_sim=30.0)
        rng_j = np.random.default_rng(c.seed + int(jv * 1000))
        seeds_j = rng_j.integers(0, 100_000, cs.n_trials).tolist()
        entry = {}
        for mode in ['tput_opt', 'aoi_opt']:
            args = [(int(s), mode, i, False, cs) for i, s in enumerate(seeds_j)]
            with Pool(c.n_workers) as pool:
                tr = pool.map(run_trial, args)
            v = np.array([r['mean_aoi'] for r in tr])
            entry[mode] = (float(v.mean()), float(v.std(ddof=1)))
        jit_res[jv] = entry
        print(f'  σ_J={jv:.3f}: AoI-Opt={entry["aoi_opt"][0]:.4f}  Tput={entry["tput_opt"][0]:.4f}')
    return jit_res

print('Sensitivity/robustness runners OK')


Runners OK
Sensitivity/robustness runners OK


In [8]:
# ══════════════════════════════════════════════════════════════════
# RUN SIMULATION  (run this cell first — takes ~90 s with NGSIM)
# ══════════════════════════════════════════════════════════════════
global _NGSIM_DF, _HAS_REAL

c = Cfg()
os.makedirs(c.outdir, exist_ok=True)

mob_label = 'Synthetic'
if os.path.exists(c.dataset):
    try:
        print(f"Loading NGSIM from '{c.dataset}' ...")
        _NGSIM_DF = load_ngsim(c.dataset)
        _HAS_REAL = True
        mob_label = 'NGSIM US-101'
        print(f"  ✓ {len(_NGSIM_DF):,} rows, {_NGSIM_DF['Vehicle_ID'].nunique()} vehicles")
    except Exception as e:
        print(f'  ✗ Failed: {e}  → using synthetic mobility')
else:
    print('  dataset.txt not found → using synthetic IDM-like mobility')

print(f"\n{'='*60}")
print(f'AoI-Optimal UAV Relay · IEEE WCL')
print(f'  Mobility: {mob_label}')
print(f'  SEED={c.seed}  N_TRIALS={c.n_trials}  N_VEH={c.n_veh}  K={c.k_relay}')
print(f'  Nakagami-m: V2V m={c.nak_v2v}  U2V m={c.nak_u2v}')
print(f"{'='*60}\n")

t0  = time.time()
res = run_all(c)
S   = summarise(res, c)
sc  = run_scale(c)

sens  = run_sensitivity(c)
ngrob = run_ngsim_robust(c)
jit   = run_jitter_sweep(c)

print(f'\nTotal time: {time.time()-t0:.1f} s')
print('Simulation complete — run individual figure cells below')

Loading NGSIM from 'dataset.txt' ...
  ✓ 1,180,598 rows, 595 vehicles

AoI-Optimal UAV Relay · IEEE WCL
  Mobility: NGSIM US-101
  SEED=42  N_TRIALS=50  N_VEH=8  K=6
  Nakagami-m: V2V m=1.5  U2V m=3.0

  No Relay                AoI=0.6490±0.4872  Risk=0.0728  DR=82.3%
  Random                  AoI=0.2175±0.0101  Risk=0.0165  DR=90.5%
  Round-Robin             AoI=0.2146±0.0085  Risk=0.0156  DR=90.4%
  Tput-Opt                AoI=0.2310±0.0295  Risk=0.0205  DR=90.5%
  Age-Threshold           AoI=0.2153±0.0074  Risk=0.0155  DR=88.3%
  AoI-Opt (Ours)          AoI=0.2111±0.0058  Risk=0.0146  DR=90.5%

Scalability sweep...
  N= 4: AoI-Opt=0.2009s  Tput-Opt=0.2009s  gain=0.0%
  N= 6: AoI-Opt=0.2041s  Tput-Opt=0.2060s  gain=1.0%
  N= 8: AoI-Opt=0.2120s  Tput-Opt=0.2320s  gain=8.6%
  N=10: AoI-Opt=0.2212s  Tput-Opt=0.2773s  gain=20.2%
  N=12: AoI-Opt=0.2338s  Tput-Opt=0.3831s  gain=39.0%
  N=14: AoI-Opt=0.2300s  Tput-Opt=0.3452s  gain=33.4%

Sensitivity sweep (alt × K)...
  h30_K4: AoI-Opt=0.2

In [17]:
# ══════════════════════════════════════════════════════════════════
# FIGURE A — AoI Evolution  [95% CI]
# ══════════════════════════════════════════════════════════════════
plt.rcParams.update(IEEE_RC)
sm    = lambda x: uniform_filter1d(np.asarray(x, float), 10)
t_ax  = np.arange(c.T) * c.dt
t95   = stats.t.ppf(0.975, c.n_trials - 1)
tst   = c.t_sim * 0.40
iant  = int(c.t_sim * 0.80 / c.dt)
RED   = '#B71C1C'

def ci_tr(mode, key):
    mat = np.stack([r[key] for r in res[mode]])
    mu  = sm(mat.mean(0))
    hw  = t95 * mat.std(0, ddof=1) / np.sqrt(c.n_trials)
    return mu, sm(mu - hw), sm(mu + hw)

fig, ax = plt.subplots(figsize=(9, 6))
_sax(ax)

for mode in MODES:
    lw, ls, al = STYLES[mode]
    mu, lo, hi = ci_tr(mode, 'aoi_tr')
    ax.plot(t_ax, mu, ls, color=COLORS[mode], lw=lw, alpha=al, label=LABELS[mode])
    ax.fill_between(t_ax, lo, hi, color=COLORS[mode], alpha=0.10)

ax.axhline(c.safety_thr, color=RED, lw=1.8, ls=':', zorder=5)
ax.text(0.04, 0.97, f'θ = {c.safety_thr} s', fontsize=FS['note'],
        color=RED, va='top', transform=ax.transAxes, fontweight='bold')

mu_ao, *_ = ci_tr('aoi_opt', 'aoi_tr')
van = mu_ao[min(iant, len(mu_ao)-1)]
ax.annotate(f"↓{S['red_mean_aoi']:.1f}%\nvs. Tput-Opt",
    xy=(t_ax[iant], van), xycoords='data',
    xytext=(0.24, 0.60), textcoords='axes fraction',
    fontsize=FS['annot'], color=COLORS['aoi_opt'], fontweight='bold', ha='center',
    arrowprops=dict(arrowstyle='-|>', color=COLORS['aoi_opt'], lw=1.8,
                    mutation_scale=14, connectionstyle='arc3,rad=-0.15'),
    bbox=dict(fc='white', ec=COLORS['aoi_opt'], boxstyle='round,pad=0.4', lw=1.6),
    zorder=10)

ax.set_xlabel('Time (s)')
ax.set_ylabel('Mean AoI (s)')
ax.set_title('AoI Evolution  [95% CI]', loc='left', pad=10)
ax.set_xlim(tst, c.t_sim)
ax.set_ylim(bottom=0)
ax.legend(loc='upper right', fontsize=FS['legend'], ncol=1, framealpha=0.92)

fig.tight_layout()
_save(fig, c.outdir, 'fig_a')

  ✓ fig_a.pdf / .png  →  figures/


In [16]:
# ══════════════════════════════════════════════════════════════════
# FIGURE B — Peak AoI Reliability (CDF)
# ══════════════════════════════════════════════════════════════════
plt.rcParams.update(IEEE_RC)
RED = '#B71C1C'

fig, ax = plt.subplots(figsize=(9, 6))
_sax(ax)

for mode in MODES:
    lw, ls, al = STYLES[mode]
    for r in res[mode]:
        if 'peak_aoi' in r: pp = r['peak_aoi']; break
    else:
        pp = np.full(10, S[mode]['peak_p95'][0])
    xs = np.sort(pp)
    ys = np.arange(1, len(xs)+1) / len(xs)
    ax.plot(xs, ys, ls, color=COLORS[mode], lw=lw, alpha=al, label=LABELS[mode])

ax.axvline(c.safety_thr, color=RED, lw=1.8, ls=':', zorder=5)

tp95 = S['tput_opt']['peak_p95'][0]
ao95 = S['aoi_opt']['peak_p95'][0]
for v, lbl, yf, mk in [(tp95,'Tput',0.72,'tput_opt'),(ao95,'AoI-Opt',0.28,'aoi_opt')]:
    cc = COLORS[mk]
    ax.annotate(f'{lbl}\n{v:.2f} s',
        xy=(v, yf), xycoords='data', xytext=(0.77, yf), textcoords='axes fraction',
        fontsize=FS['annot'], color=cc, fontweight='bold', ha='center',
        arrowprops=dict(arrowstyle='-|>', color=cc, lw=1.6, mutation_scale=12),
        bbox=dict(fc='white', ec=cc, boxstyle='round,pad=0.35', lw=1.4), zorder=10)

ax.set_xlabel('95th-Pct AoI (s)')
ax.set_ylabel('CDF')
ax.set_title('Peak AoI Reliability (CDF)', loc='left', pad=10)
ax.set_xlim(0, min(3.0, max(tp95, ao95)*2.5))
ax.set_ylim(0, 1.04)
ax.legend(loc='lower right', fontsize=FS['legend'])

fig.tight_layout()
_save(fig, c.outdir, 'fig_b')

  ✓ fig_b.pdf / .png  →  figures/


In [15]:
# ══════════════════════════════════════════════════════════════════
# FIGURE C — Link Budget (Nakagami fading)
# ══════════════════════════════════════════════════════════════════
plt.rcParams.update(IEEE_RC)
RED   = '#B71C1C'
GREEN = '#2E7D32'

fig, ax = plt.subplots(figsize=(9, 6))
_sax(ax)

d_arr = np.linspace(5, 900, 800)
sv = snr_v2v(d_arr, c)
su = snr_u2v(d_arr, c)

l1, = ax.plot(d_arr, sv, '-',  color=COLORS['aoi_opt'],    lw=2.8, label='SNR V2V')
l2, = ax.plot(d_arr, su, '-',  color=COLORS['round_robin'], lw=2.8, label='SNR U2V')
l3  = ax.axhline(c.snr_mid, color=RED,   lw=1.8, ls='--', label='SNR threshold')
l4  = ax.axvline(c.direct_range, color=GREEN, lw=1.8, ls=':', label='Direct range')

ax.set_xlabel('Distance (m)')
ax.set_ylabel('Mean SNR (dB)')
ax.set_xlim(0, 800)
ax.set_ylim(-10, 55)

ax2 = ax.twinx()
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_linewidth(1.6)
ax2.spines['right'].set_color('#333')
ax2.tick_params(which='both', direction='in', width=1.3, labelsize=FS['tick'])
l5, = ax2.plot(d_arr, pdr_fn(sv, c), ':', color=COLORS['aoi_opt'],    lw=2.8, alpha=0.9, label='PDR V2V')
l6, = ax2.plot(d_arr, pdr_fn(su, c), ':', color=COLORS['round_robin'], lw=2.8, alpha=0.9, label='PDR U2V')
ax2.set_ylim(-0.05, 1.2)
ax2.set_ylabel('PDR', fontsize=FS['label'], fontweight='bold')

ax.set_title('Link Budget (Nakagami fading)', loc='left', pad=10)
ax.text(0.03, 0.05,
    f'V2V m={c.nak_v2v}, U2V m={c.nak_u2v}  |  '
    f'$f_c$={c.f_c/1e9:.1f} GHz  $P_{{\\rm tx}}$={c.p_tx_dbm} dBm',
    fontsize=FS['note'], color='0.45', transform=ax.transAxes, style='italic', va='bottom')
ax.legend([l1, l2, l5, l6, l3, l4], [h.get_label() for h in [l1, l2, l5, l6, l3, l4]],
          loc='upper right', ncol=2, fontsize=FS['legend'], framealpha=0.95)

fig.tight_layout()
_save(fig, c.outdir, 'fig_c')

  ✓ fig_c.pdf / .png  →  figures/


In [14]:
# ══════════════════════════════════════════════════════════════════
# FIGURE D — Collision-Risk Proxy P_risk(t)
# ══════════════════════════════════════════════════════════════════
plt.rcParams.update(IEEE_RC)
sm    = lambda x: uniform_filter1d(np.asarray(x, float), 10)
t_ax  = np.arange(c.T) * c.dt
t95   = stats.t.ppf(0.975, c.n_trials - 1)
tst   = c.t_sim * 0.40
iant  = int(c.t_sim * 0.80 / c.dt)

def ci_tr(mode, key):
    mat = np.stack([r[key] for r in res[mode]])
    mu  = sm(mat.mean(0))
    hw  = t95 * mat.std(0, ddof=1) / np.sqrt(c.n_trials)
    return mu, sm(mu - hw), sm(mu + hw)

fig, ax = plt.subplots(figsize=(9, 6))
_sax(ax)

for mode in MODES:
    lw, ls, al = STYLES[mode]
    mu, lo, hi = ci_tr(mode, 'risk_tr')
    ax.plot(t_ax, mu, ls, color=COLORS[mode], lw=lw, alpha=al, label=LABELS[mode])
    ax.fill_between(t_ax, lo, hi, color=COLORS[mode], alpha=0.10)

mu_r, *_ = ci_tr('aoi_opt', 'risk_tr')
vr = mu_r[min(iant, len(mu_r)-1)]
ax.annotate(f"↓{S['red_mean_risk']:.1f}%\nvs. Tput-Opt",
    xy=(t_ax[iant], vr), xycoords='data',
    xytext=(0.24, 0.60), textcoords='axes fraction',
    fontsize=FS['annot'], color=COLORS['aoi_opt'], fontweight='bold', ha='center',
    arrowprops=dict(arrowstyle='-|>', color=COLORS['aoi_opt'], lw=1.8,
                    mutation_scale=14, connectionstyle='arc3,rad=-0.15'),
    bbox=dict(fc='white', ec=COLORS['aoi_opt'], boxstyle='round,pad=0.4', lw=1.6),
    zorder=10)

ax.set_xlabel('Time (s)')
ax.set_ylabel(r'Mean $P_{\rm risk}$')
ax.set_title(r'Collision-Risk Proxy $P_{\rm risk}(t)$', loc='left', pad=10)
ax.set_xlim(tst, c.t_sim)
ax.set_ylim(bottom=0)
ax.text(0.03, 0.04,
    r'$P_{\rm risk}=1-e^{-\sigma_v^2\Delta^2/2r^2}$ — first-order proxy only.',
    fontsize=FS['note'], color='0.45', transform=ax.transAxes, style='italic', va='bottom')
ax.legend(loc='upper right', fontsize=FS['legend'], framealpha=0.92)

fig.tight_layout()
_save(fig, c.outdir, 'fig_d')

  ✓ fig_d.pdf / .png  →  figures/


In [10]:
# ══════════════════════════════════════════════════════════════════
# FIGURE E — Scalability: Mean AoI vs. Platoon Size
# ══════════════════════════════════════════════════════════════════
plt.rcParams.update(IEEE_RC)
RED = '#B71C1C'

fig, ax = plt.subplots(figsize=(9, 6))
_sax(ax)

nv_arr   = np.array(c.scale_nv)
sc_ls    = {'tput_opt': '--', 'age_threshold': '-.', 'aoi_opt': '-'}
sc_modes = ['tput_opt', 'age_threshold', 'aoi_opt']

for mode in sc_modes:
    mus = np.array([sc[nv][mode][0] for nv in c.scale_nv])
    cis = np.array([sc[nv][mode][1] for nv in c.scale_nv])
    lw, _, al = STYLES[mode]
    ax.plot(nv_arr, mus, sc_ls[mode], color=COLORS[mode], lw=lw, alpha=al, label=LABELS[mode])
    ax.fill_between(nv_arr, mus - cis, mus + cis, color=COLORS[mode], alpha=0.15)

ax.axhline(c.safety_thr, color=RED, lw=1.8, ls=':', zorder=5)
ax.text(max(c.scale_nv)+0.25, c.safety_thr, f'θ={c.safety_thr} s',
        fontsize=FS['note']+1, color=RED, va='center', clip_on=False)
ax.axvline(c.n_veh, color='0.5', lw=1.6, ls=':', alpha=0.7)
ax.text(c.n_veh+0.2, sc[c.n_veh]['aoi_opt'][0]+0.005,
        f'$N$={c.n_veh}', fontsize=FS['note']+1, color='0.3', va='bottom')

ax.set_xlabel(r'Number of Vehicles $N$')
ax.set_ylabel('Mean AoI (s)')
ax.set_title('Scalability — Mean AoI vs. Platoon Size [95% CI]', loc='left', pad=10)
ax.set_xlim(min(c.scale_nv)-0.5, max(c.scale_nv)+0.5)
ax.set_ylim(bottom=0)
ax.xaxis.set_major_locator(MultipleLocator(2))
ax.text(0.02, 0.97,
        f'{c.scale_tr} trials/point · $T_{{\\rm sim}}$={c.scale_tsim} s · {mob_label}',
        fontsize=FS['note'], color='0.45', transform=ax.transAxes, va='top', style='italic')
ax.legend(loc='upper left', fontsize=FS['legend'])

fig.tight_layout()
_save(fig, c.outdir, 'fig_e')

print('\nAll figures saved:')
for fn in ['fig_a','fig_b','fig_c','fig_d','fig_e']:
    for ext in ['.pdf','.png']:
        p = os.path.join(c.outdir, fn+ext)
        exists = '✓' if os.path.exists(p) else '✗'
        print(f'  {exists}  {p}')

  ✓ fig_e.pdf / .png  →  figures/

All figures saved:
  ✗  figures/fig_a.pdf
  ✗  figures/fig_a.png
  ✗  figures/fig_b.pdf
  ✗  figures/fig_b.png
  ✗  figures/fig_c.pdf
  ✗  figures/fig_c.png
  ✗  figures/fig_d.pdf
  ✗  figures/fig_d.png
  ✓  figures/fig_e.pdf
  ✓  figures/fig_e.png


In [13]:
# ══════════════════════════════════════════════════════════════════
# FIGURE F — Sensitivity & Robustness  (3 subplots)
# ══════════════════════════════════════════════════════════════════
plt.rcParams.update(IEEE_RC)

figf, axs = plt.subplots(1, 3, figsize=(18, 5))
for ax in axs: _sax(ax)

# ── (a) Jitter sweep ──────────────────────────────────────────────
jit_vals = sorted(jit.keys())
for mode, mk in [('aoi_opt', COLORS['aoi_opt']), ('tput_opt', COLORS['tput_opt'])]:
    mus = [jit[j][mode][0] for j in jit_vals]
    sds = [jit[j][mode][1] for j in jit_vals]
    lw, ls, al = STYLES[mode]
    axs[0].plot(jit_vals, mus, ls + 'o', color=mk, lw=lw, alpha=al,
                ms=8, label=LABELS[mode])
    axs[0].fill_between(jit_vals,
                        [m - s for m, s in zip(mus, sds)],
                        [m + s for m, s in zip(mus, sds)],
                        color=mk, alpha=0.12)
# annotate max gap
ao0 = jit[0.0]['aoi_opt'][0]; tp0 = jit[0.0]['tput_opt'][0]
ao5 = jit[0.05]['aoi_opt'][0]; tp5 = jit[0.05]['tput_opt'][0]
gap0 = 100*(tp0-ao0)/tp0; gap5 = 100*(tp5-ao5)/tp5
axs[0].text(0.05, 0.97,
    f'Gap: {gap0:.1f}% → {gap5:.1f}%  (σ_J 0→0.05 s)',
    transform=axs[0].transAxes, fontsize=FS['note'], color='0.35',
    va='top', style='italic')
axs[0].set_title('(a) Jitter σ_J Sensitivity', loc='left', pad=8)
axs[0].set_xlabel('Jitter σ_J (s)')
axs[0].set_ylabel('Mean AoI (s)')
axs[0].legend(fontsize=FS['legend'] - 1)

# ── (b) Alt × K heatmap-style lines ──────────────────────────────
alts_plot = [30, 60, 120]
ks_plot   = [4, 6, 8]
k_colors  = ['#1565C0', '#00838F', '#558B2F']
for ki, k in enumerate(ks_plot):
    mus_ao = [sens[f'h{int(a)}_K{k}']['aoi_opt'][0]  for a in alts_plot]
    mus_tp = [sens[f'h{int(a)}_K{k}']['tput_opt'][0] for a in alts_plot]
    axs[1].plot(alts_plot, mus_ao, 'o-',  color=k_colors[ki], lw=2.4, ms=7,
                label=f'AoI-Opt K={k}')
    axs[1].plot(alts_plot, mus_tp, 's--', color=k_colors[ki], lw=1.8, ms=6,
                alpha=0.6, label=f'Tput-Opt K={k}')
axs[1].set_title('(b) Alt × K Sensitivity', loc='left', pad=8)
axs[1].set_xlabel('UAV Altitude h_u (m)')
axs[1].set_ylabel('Mean AoI (s)')
axs[1].legend(fontsize=FS['legend'] - 2, ncol=2)

# ── (c) NGSIM smoothing robustness ───────────────────────────────
if ngrob:
    sw_keys  = sorted(ngrob.keys(), key=lambda x: int(x[2:]))
    sw_ticks = [int(k[2:]) for k in sw_keys]
    ao_mus   = [ngrob[k]['aoi_opt'][0]  for k in sw_keys]
    tp_mus   = [ngrob[k]['tput_opt'][0] for k in sw_keys]
    ao_sds   = [ngrob[k]['aoi_opt'][1]  for k in sw_keys]
    tp_sds   = [ngrob[k]['tput_opt'][1] for k in sw_keys]
    x = np.arange(len(sw_ticks))
    w = 0.35
    bars_ao = axs[2].bar(x - w/2, ao_mus, w,
                          color=COLORS['aoi_opt'], alpha=0.85,
                          yerr=ao_sds, capsize=5, label=LABELS['aoi_opt'])
    bars_tp = axs[2].bar(x + w/2, tp_mus, w,
                          color=COLORS['tput_opt'], alpha=0.85,
                          yerr=tp_sds, capsize=5, label=LABELS['tput_opt'])
    axs[2].set_xticks(x)
    axs[2].set_xticklabels([f'sw={s}' for s in sw_ticks])
    # annotate max AoI variance across windows
    var_pct = 100*(max(ao_mus) - min(ao_mus)) / max(ao_mus)
    axs[2].text(0.05, 0.97,
        f'AoI-Opt var across windows: {var_pct:.1f}%',
        transform=axs[2].transAxes, fontsize=FS['note'], color='0.35',
        va='top', style='italic')
else:
    axs[2].text(0.5, 0.5, 'No NGSIM data\n(synthetic mobility only)',
                ha='center', va='center', transform=axs[2].transAxes,
                fontsize=FS['legend'], color='0.5')
axs[2].set_title('(c) NGSIM Smoothing Robustness', loc='left', pad=8)
axs[2].set_xlabel('Smoothing Window')
axs[2].set_ylabel('Mean AoI (s)')
axs[2].legend(fontsize=FS['legend'] - 1)

figf.suptitle('Sensitivity & Robustness Analysis', fontsize=FS['title'],
              fontweight='bold', y=1.01)
figf.tight_layout()
_save(figf, c.outdir, 'fig_f')


  ✓ fig_f.pdf / .png  →  figures/


In [9]:
# Add to summarise(), after the t-test loop:
for k, label in [('mean_aoi', 'aoi'), ('mean_risk', 'risk')]:
    a = np.array([r[k] for r in res['aoi_opt']])
    b = np.array([r[k] for r in res['tput_opt']])
    diff = b - a  # paired differences
    pooled_std = np.sqrt((a.std(ddof=1)**2 + b.std(ddof=1)**2) / 2)
    S[f'cohens_d_{label}_vs_tput'] = float((b.mean() - a.mean()) / pooled_std)
    print(f"  Cohen's d ({label} vs Tput-Opt): {S[f'cohens_d_{label}_vs_tput']:.2f}")

  Cohen's d (aoi vs Tput-Opt): 0.94
  Cohen's d (risk vs Tput-Opt): 1.17
